In [5]:
import pandas as pd
import numpy as np
from google.colab import files
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, RocCurveDisplay

# 1. S - SAMPLE (Amostragem)
# ----------------------------------------------------------------
print("Aguardando upload do arquivo 'abt_churn.csv'...")
uploaded = files.upload()
file_name = list(uploaded.keys())[0]
df = pd.read_csv(file_name)

# Configurações de colunas baseadas no seu arquivo
col_ref = 'dtRef'
target = 'flagChurn'
col_id = 'idUsuario'

df[col_ref] = pd.to_datetime(df[col_ref])

# Out-of-Time (OOT): Reserva do último mês
data_oot = df[col_ref].max()
base_oot = df[df[col_ref] == data_oot].copy()
base_modelagem = df[df[col_ref] < data_oot].copy()

# Split Treino (80%) e Teste (20%) com Estratificação [cite: 12, 13]
X = base_modelagem.drop(columns=[target, col_ref, col_id])
y = base_modelagem[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"\n[S] Amostragem concluída. Treino: {len(X_train)}, Teste: {len(X_test)}, OOT: {len(base_oot)}")

# 2. E - EXPLORE (Exploração)
# ----------------------------------------------------------------
# Verificação de nulos apenas no treino [cite: 17, 20]
print("\n[E] Verificação de Nulos no Treino:")
print(X_train.isnull().sum().sort_values(ascending=False).head(5))

# Feature Importance Inicial (Árvore de Decisão Simples) [cite: 23, 24]
dt_initial = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_initial.fit(X_train.fillna(0), y_train) # Fillna temporário para exploração

importances = pd.Series(dt_initial.feature_importances_, index=X_train.columns)
print("\n[E] Top 5 Variáveis (Feature Importance Inicial):")
print(importances.sort_values(ascending=False).head(5))

# 3. M - MODIFY (Modificação) & 4. M - MODEL (Modelagem)
# ----------------------------------------------------------------
# Identificando colunas numéricas e categóricas
num_features = X_train.select_dtypes(include=['int64', 'float64']).columns
cat_features = X_train.select_dtypes(include=['object']).columns

# Pipeline de Transformação
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')), # Tratamento de missings [cite: 21]
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')) # Encoding [cite: 30]
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_features),
    ('cat', cat_transformer, cat_features)
])

# Pipeline Final com Modelo Random Forest [cite: 38]
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

# Tuning de Hiperparâmetros via GridSearchCV [cite: 39, 42]
param_grid = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [5, 10],
    'classifier__min_samples_split': [2, 5]
}

print("\n[M] Iniciando Modelagem e Tuning (GridSearch)...")
grid_search = GridSearchCV(model_pipeline, param_grid, cv=3, scoring='roc_auc', n_jobs=-1)
grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
print(f"Melhores parâmetros: {grid_search.best_params_}")

# 5. A - ASSESS (Avaliação)
# ----------------------------------------------------------------
def avaliar_modelo(nome, X_eval, y_eval):
    y_pred = best_model.predict(X_eval)
    y_proba = best_model.predict_proba(X_eval)[:, 1]
    auc = roc_auc_score(y_eval, y_proba)
    acc = accuracy_score(y_eval, y_pred)
    print(f"Métricas {nome} -> ROC AUC: {auc:.4f} | Acurácia: {acc:.4f}")
    return auc, y_proba

print("\n[A] Resultados Finais:")
auc_train, _ = avaliar_modelo("Treino", X_train, y_train)
auc_test, y_proba_test = avaliar_modelo("Teste", X_test, y_test)
auc_oot, _ = avaliar_modelo("OOT", base_oot.drop(columns=[target, col_ref, col_id]), base_oot[target])

# Análise de Negócio (Lift/Gains) [cite: 51, 52]
test_results = pd.DataFrame({'prob': y_proba_test, 'real': y_test})
test_results = test_results.sort_values(by='prob', ascending=False)
test_results['cum_churn'] = test_results['real'].cumsum() / test_results['real'].sum()
test_results['percentil'] = np.arange(len(test_results)) / len(test_results)

print("\n[A] Ganho de Negócio (Capture Rate):")
for p in [0.1, 0.2, 0.3]:
    gain = test_results[test_results['percentil'] <= p]['cum_churn'].max()
    print(f"Abordando os top {int(p*100)}% de risco, capturamos {gain:.2%} dos churners.")

# Serialização do modelo [cite: 56]
import pickle
with open('modelo_churn_semma.pkl', 'wb') as f:
    pickle.dump(best_model, f)
print("\nModelo salvo como 'modelo_churn_semma.pkl'")

Aguardando upload do arquivo 'abt_churn.csv'...


Saving abt_churn.csv to abt_churn (2).csv

[S] Amostragem concluída. Treino: 4154, Teste: 1039, OOT: 303

[E] Verificação de Nulos no Treino:
qtdeTransacoes         0
qtdeDias               0
mediaTransacoesDias    0
saldoPontos            0
qtdePontosPos          0
dtype: int64

[E] Top 5 Variáveis (Feature Importance Inicial):
qtdeDiasD14                0.620636
propAvgQtdeDias            0.119771
qtdeDiasUltimaTransacao    0.056484
qtdeTransacoesD7           0.042025
qtdePontosPosD7            0.035461
dtype: float64

[M] Iniciando Modelagem e Tuning (GridSearch)...
Melhores parâmetros: {'classifier__max_depth': 5, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 200}

[A] Resultados Finais:
Métricas Treino -> ROC AUC: 0.8433 | Acurácia: 0.7595
Métricas Teste -> ROC AUC: 0.8290 | Acurácia: 0.7526
Métricas OOT -> ROC AUC: 0.8428 | Acurácia: 0.7723

[A] Ganho de Negócio (Capture Rate):
Abordando os top 10% de risco, capturamos 18.48% dos churners.
Abordando os top 20% d